# Fine-tuning de Ministral 3 3B · Asistente offline para apagones

**Objetivo:** enseñar el *estilo* (texto plano, 1–3 frases, copiar el dato calculado, negarse y derivar en trampas). Los datos los sigue poniendo el RAG.

1. *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (gratis).
2. Ejecuta las celdas en orden. Te pedirá subir `train.jsonl` y `valid.jsonl` (de `finetune/data/`).
3. Al final descarga el GGUF (y el adaptador LoRA de respaldo).

Base: plantilla de Unsloth para Ministral 3 (visión), con la parte de visión congelada y datos solo de texto.
Si alguna celda falla por versiones, abre el [cuaderno oficial de Unsloth para Ministral 3 3B](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Ministral_3_VL_(3B)_Vision.ipynb), ejecuta su instalación y sustituye sus celdas de datos por las de este cuaderno.

In [ ]:
%%capture
!pip install unsloth
# Ministral 3 necesita transformers v5 y la última versión de unsloth (docs de Unsloth)
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install --upgrade "transformers>=5" 

In [ ]:
from google.colab import files
up = files.upload()   # sube train.jsonl y valid.jsonl
print(list(up))

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Ministral-3-3B-Instruct-2512",
    load_in_4bit = True,                     # QLoRA: cabe en una T4
    use_gradient_checkpointing = "unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,      # no tocamos la visión
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    random_state = 42,
)

In [ ]:
# Arreglo del tokenizador: el de Ministral 3 trae un patrón (regex) incorrecto y partiría el texto
# distinto de como lo hará llama.cpp en Ollama y en la app. Se recarga con fix_mistral_regex=True.
from transformers import AutoProcessor, AutoTokenizer
BASE = "unsloth/Ministral-3-3B-Instruct-2512"
viejo = tokenizer
try:
    tokenizer = AutoProcessor.from_pretrained(BASE, fix_mistral_regex=True)
except TypeError:
    tokenizer = viejo
    tokenizer.tokenizer = AutoTokenizer.from_pretrained(BASE, fix_mistral_regex=True)

t = "¿Cuántas horas aguanta la nevera?  Llevo 3h sin luz y no he abierto el congelador."
tok_v = getattr(viejo, "tokenizer", viejo); tok_n = getattr(tokenizer, "tokenizer", tokenizer)
print("antes:", len(tok_v(t)["input_ids"]), "tokens · ahora:", len(tok_n(t)["input_ids"]), "tokens")
print(type(tokenizer).__name__)

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "train.jsonl", "validation": "valid.jsonl"})
print(ds)
print(ds["train"][0]["messages"][2]["content"][0]["text"])

## Antes de entrenar: cómo responde ahora

In [ ]:
def generar(msgs, max_new_tokens=200):
    FastVisionModel.for_inference(model)
    prompt = tokenizer.apply_chat_template(msgs[:2], add_generation_prompt=True)
    inputs = tokenizer(text=prompt, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.15, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

muestras = [ds["validation"][i]["messages"] for i in range(min(5, len(ds["validation"])))]
antes = [generar(m) for m in muestras]
for m, a in zip(muestras, antes):
    print("P:", m[1]["content"][0]["text"].split("PREGUNTA:")[-1].strip())
    print("ANTES:", a, f"({len(a.split())} palabras)")
    print("IDEAL:", m[2]["content"][0]["text"], "\n")

## Entrenamiento

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = ds["train"],
    eval_dataset = ds["validation"],
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 10,
        logging_steps = 10,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,                   # prompt con reglas + 6 trozos de guías ≈ 1.300 tokens
    ),
)
stats = trainer.train()
print(stats)

## Después de entrenar: mismas preguntas

In [ ]:
for m, a in zip(muestras, antes):
    d = generar(m)
    print("P:", m[1]["content"][0]["text"].split("PREGUNTA:")[-1].strip())
    print(f"ANTES ({len(a.split())} pal.):", a)
    print(f"DESPUÉS ({len(d.split())} pal.):", d, "\n")

## Guardar y descargar

Primero el adaptador LoRA (respaldo, ligero). Después el GGUF Q4_K_M para Ollama y para la app.

In [ ]:
model.save_pretrained("ministral3-apagon-lora")
tokenizer.save_pretrained("ministral3-apagon-lora")
!zip -rq ministral3-apagon-lora.zip ministral3-apagon-lora
files.download("ministral3-apagon-lora.zip")

In [ ]:
# GGUF cuantizado (el que irá en la app). Si esta celda falla con el modelo de visión, usa la siguiente.
model.save_pretrained_gguf("ministral3-apagon", tokenizer, quantization_method = "q4_k_m")
!ls -lh ministral3-apagon*

In [ ]:
# Alternativa: modelo fusionado en 16 bits; luego se convierte a GGUF en el Mac con llama.cpp (ver finetune/README.md)
# model.save_pretrained_merged("ministral3-apagon-merged", tokenizer, save_method = "merged_16bit")
# !zip -rq merged.zip ministral3-apagon-merged && files.download("merged.zip")

In [ ]:
import glob
for f in glob.glob("ministral3-apagon*/*.gguf") + glob.glob("*.gguf"):
    print("descargando", f); files.download(f)